# Telco Customer Churn: Retention & Risk Analysis
**Goal:** measure baseline churn, build retention curves by contract, payment method and monthly charges, and find the high-risk segments worth targeting.

**Flow:** 1. Load & clean → 2. Baseline churn → 3. Tenure cohorts & contract type → 4. Retention curves (Kaplan-Meier) → 5. Payment & charges → 6. Churn heatmaps → 7. High-risk segments → 8. Churn probability model → 9. Takeaways

*Dataset: IBM Telco Customer Churn (7,043 customers). Place `Telco-Customer-Churn.csv` next to this notebook.*

In [1]:
import pandas as pd, numpy as np
import plotly.express as px
import plotly.graph_objects as go
pd.options.display.float_format = "{:,.2f}".format

## 1. Load & clean

In [2]:
df = pd.read_csv("Telco-Customer-Churn.csv")
print("Shape:", df.shape, "| nulls:", df.isna().sum().sum(), "| duplicate IDs:", df["customerID"].duplicated().sum())

# TotalCharges is stored as text; 11 blanks are brand-new customers (tenure = 0) who haven't been billed yet
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")
print("Blank TotalCharges:", df["TotalCharges"].isna().sum(), "| their tenure:", df.loc[df["TotalCharges"].isna(), "tenure"].unique())
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Churn flag + readable labels
df["Churned"] = (df["Churn"] == "Yes").astype(int)
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

# Tenure cohorts (months) and monthly-charge bands
df["Tenure Cohort"] = pd.cut(df["tenure"], [-1, 12, 24, 36, 48, 60, 72],
                             labels=["0-12", "13-24", "25-36", "37-48", "49-60", "61-72"])
df["Charge Band"] = pd.cut(df["MonthlyCharges"], [0, 35, 55, 75, 95, 130],
                           labels=["<$35", "$35-55", "$55-75", "$75-95", "$95+"])
df.head()

Shape: (7043, 21) | nulls: 0 | duplicate IDs: 0
Blank TotalCharges: 11 | their tenure: [0]


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Churned,Tenure Cohort,Charge Band
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,...,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0,0-12,<$35
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,...,No,One year,No,Mailed check,56.95,"1,889.50",No,0,25-36,$55-75
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,0-12,$35-55
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,...,No,One year,No,Bank transfer (automatic),42.30,"1,840.75",No,0,37-48,$35-55
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,...,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,0-12,$55-75


## 2. Baseline churn rate

In [3]:
total, churned = len(df), df["Churned"].sum()
baseline = churned / total
print(f"Churn Rate = {churned:,} / {total:,} = {baseline:.1%}")
print(f"Avg monthly charge: churned ${df.loc[df.Churned==1,'MonthlyCharges'].mean():.2f} vs retained ${df.loc[df.Churned==0,'MonthlyCharges'].mean():.2f}")

def churn_by(col):
    g = df.groupby(col, observed=True).agg(Customers=("Churned", "size"), Churned=("Churned", "sum"), ChurnRate=("Churned", "mean"))
    g["ChurnRate"] *= 100
    return g

def bar(col, title=None):
    g = churn_by(col).reset_index()
    fig = px.bar(g, x=col, y="ChurnRate", text=g["ChurnRate"].map("{:.1f}%".format), color="ChurnRate",
                 color_continuous_scale="Reds", title=title or f"Churn rate by {col}")
    fig.add_hline(y=baseline*100, line_dash="dash", annotation_text=f"Baseline {baseline:.1%}")
    fig.update_layout(coloraxis_showscale=False, yaxis_title="Churn rate (%)", height=400)
    fig.show()

Churn Rate = 1,869 / 7,043 = 26.5%
Avg monthly charge: churned $74.44 vs retained $61.27


## 3. Tenure cohorts & contract type

In [4]:
display(churn_by("Tenure Cohort"))
bar("Tenure Cohort")
display(churn_by("Contract"))
bar("Contract")

,Customers,Churned,ChurnRate
Tenure Cohort,,,
0-12,2186,1037,47.44
13-24,1024,294,28.71
25-36,832,180,21.63
37-48,762,145,19.03
49-60,832,120,14.42
61-72,1407,93,6.61


,Customers,Churned,ChurnRate
Contract,,,
Month-to-month,3875,1655,42.71
One year,1473,166,11.27
Two year,1695,48,2.83


## 4. Retention curves (Kaplan-Meier)
A retention curve shows the share of customers still active after *t* months. Customers who haven't churned are treated as still active (censored) at their current tenure.

In [5]:
def km_curve(sub):
    t = np.arange(0, 73)
    surv, s = [1.0], 1.0
    for m in t[1:]:
        at_risk = (sub["tenure"] >= m).sum()
        events = ((sub["tenure"] == m) & (sub["Churned"] == 1)).sum()
        s *= 1 - events / at_risk if at_risk else 1
        surv.append(s)
    return t, np.array(surv)

def km_plot(col, title):
    fig = go.Figure()
    rows = []
    for name, sub in df.groupby(col, observed=True):
        t, s = km_curve(sub)
        fig.add_trace(go.Scatter(x=t, y=s*100, mode="lines", name=str(name)))
        rows.append({col: name, "12m": s[12]*100, "24m": s[24]*100, "36m": s[36]*100, "60m": s[60]*100})
    fig.update_layout(title=title, xaxis_title="Months since signup", yaxis_title="Customers retained (%)", height=450, yaxis_range=[0, 102])
    fig.show()
    display(pd.DataFrame(rows).set_index(col).round(1))

km_plot("Contract", "Retention curve by contract type")
km_plot("PaymentMethod", "Retention curve by payment method")
km_plot("Charge Band", "Retention curve by monthly charge band")

,12m,24m,36m,60m
Contract,,,,
Month-to-month,70.30,58.60,49.10,29.70
One year,99.10,97.80,95.90,83.20
Two year,100.00,100.00,99.90,98.60


,12m,24m,36m,60m
PaymentMethod,,,,
Bank transfer (automatic),93.10,90.20,87.80,80.40
Credit card (automatic),94.50,91.30,88.70,83.10
Electronic check,72.70,62.40,55.30,41.30
Mailed check,82.80,79.70,76.90,73.20


,12m,24m,36m,60m
Charge Band,,,,
<$35,90.40,89.10,88.10,85.70
$35-55,77.30,71.20,69.70,60.40
$55-75,80.30,74.80,71.70,65.90
$75-95,79.60,71.30,64.50,57.20
$95+,92.40,87.00,82.40,68.50


## 5. Payment method & monthly charges

In [6]:
display(churn_by("PaymentMethod")); bar("PaymentMethod")
display(churn_by("Charge Band")); bar("Charge Band")
display(churn_by("PaperlessBilling"))

,Customers,Churned,ChurnRate
PaymentMethod,,,
Bank transfer (automatic),1544,258,16.71
Credit card (automatic),1522,232,15.24
Electronic check,2365,1071,45.29
Mailed check,1612,308,19.11


,Customers,Churned,ChurnRate
Charge Band,,,
<$35,1735,189,10.89
$35-55,897,251,27.98
$55-75,1291,348,26.96
$75-95,1825,663,36.33
$95+,1295,418,32.28


,Customers,Churned,ChurnRate
PaperlessBilling,,,
No,2872,469,16.33
Yes,4171,1400,33.57


## 6. Churn heatmaps

In [7]:
def heat(rows, cols, title):
    p = df.pivot_table(index=rows, columns=cols, values="Churned", aggfunc="mean", observed=True) * 100
    fig = px.imshow(p, text_auto=".0f", color_continuous_scale="RdYlGn_r", aspect="auto", title=title,
                    labels=dict(color="Churn %"))
    fig.update_layout(height=420); fig.show()

heat("Contract", "Tenure Cohort", "Churn % by contract x tenure cohort")
heat("InternetService", "Contract", "Churn % by internet service x contract")
heat(["InternetService", "TechSupport"], "Contract", "Churn % by service tier, tech support and contract")
heat("PaymentMethod", "Contract", "Churn % by payment method x contract")

## 7. High-risk segments

In [8]:
segments = {
    "Fiber optic, no tech support":                    (df.InternetService == "Fiber optic") & (df.TechSupport == "No"),
    "Fiber, no tech support, month-to-month":          (df.InternetService == "Fiber optic") & (df.TechSupport == "No") & (df.Contract == "Month-to-month"),
    "Month-to-month, tenure <= 12 months":            (df.Contract == "Month-to-month") & (df.tenure <= 12),
    "Month-to-month + electronic check":              (df.Contract == "Month-to-month") & (df.PaymentMethod == "Electronic check"),
    "Senior citizens":                                 df.SeniorCitizen == "Yes",
    "No online security":                              df.OnlineSecurity == "No",
    "Two-year contract (low-risk benchmark)":          df.Contract == "Two year",
}
rows = []
for name, mask in segments.items():
    sub = df[mask]
    rows.append({"Segment": name, "Customers": len(sub), "Churn rate %": sub.Churned.mean()*100,
                 "Lift vs baseline": sub.Churned.mean()/baseline, "Share of all churners %": sub.Churned.sum()/churned*100})
risk = pd.DataFrame(rows).set_index("Segment").sort_values("Churn rate %", ascending=False)
risk

,Customers,Churn rate %,Lift vs baseline,Share of all churners %
Segment,,,,
"Fiber, no tech support, month-to-month",1796,57.52,2.17,55.27
Month-to-month + electronic check,1850,53.73,2.02,53.18
"Month-to-month, tenure <= 12 months",1994,51.35,1.94,54.79
"Fiber optic, no tech support",2230,49.37,1.86,58.91
No online security,3498,41.77,1.57,78.17
Senior citizens,1142,41.68,1.57,25.47
Two-year contract (low-risk benchmark),1695,2.83,0.11,2.57


## 8. Churn probability model
A simple logistic regression turns the segment patterns into a per-customer churn probability, so customers can be ranked and tiered.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

feat = ["tenure", "MonthlyCharges", "Contract", "PaymentMethod", "InternetService", "TechSupport", "OnlineSecurity", "PaperlessBilling", "SeniorCitizen"]
X = pd.get_dummies(df[feat], drop_first=True).astype(float)
X[["tenure", "MonthlyCharges"]] = (X[["tenure", "MonthlyCharges"]] - X[["tenure", "MonthlyCharges"]].mean()) / X[["tenure", "MonthlyCharges"]].std()
y = df["Churned"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print(f"Test ROC-AUC: {roc_auc_score(yte, model.predict_proba(Xte)[:, 1]):.3f}")

df["Churn Probability"] = model.predict_proba(X)[:, 1]
df["Risk Tier"] = pd.cut(df["Churn Probability"], [0, 0.2, 0.5, 1.0], labels=["Low (<20%)", "Medium (20-50%)", "High (>50%)"])
tier = df.groupby("Risk Tier", observed=True).agg(Customers=("Churned", "size"), ActualChurn=("Churned", "mean"))
tier["ActualChurn"] *= 100
tier

Test ROC-AUC: 0.841


,Customers,ActualChurn
Risk Tier,,
Low (<20%),3661,7.32
Medium (20-50%),1870,33.05
High (>50%),1512,65.01


In [10]:
coef = pd.Series(model.coef_[0], index=X.columns).sort_values()
px.bar(coef, orientation="h", title="What pushes churn up or down (logistic regression coefficients)",
       labels={"value": "Coefficient", "index": ""}).update_layout(showlegend=False, height=500).show()

## 9. Takeaways

1. **Baseline churn is 26.5%** (1,869 of 7,043 customers), and the first year is where it happens: 47.4% of customers in their first 12 months churn, falling steadily to 6.6% for customers beyond 5 years.

2. **Contract type is the strongest lever.** Month-to-month customers churn at 42.7%, versus 11.3% on one-year and 2.8% on two-year contracts. *Action: offer a discount or bundle to move month-to-month customers onto annual plans, ideally before month 12.*

3. **Fiber optic customers without tech support are the biggest at-risk group.** They churn at 49.4% (2,230 customers), rising to 57.5% if they are also month-to-month. With tech support the same fiber customers churn at 22.6%. *Action: bundle tech support (or a free trial) for new fiber signups.*

4. **Electronic-check payers churn at 45.3%**, about 3x the 15-17% of automatic bank or card payers. *Action: nudge customers to autopay with a small incentive.* This is likely a signal of weaker commitment rather than a cause, so treat it as a targeting flag.

5. **Higher bills mean higher churn.** Churners pay about $74 per month on average, versus $61 for retained customers. Customers under $35 churn at only 10.9%, while the $75-95 band churns at 36.3%. *Action: review pricing and value perception for the $75+ tiers.*

6. **Early-life month-to-month customers are where most churn lives.** Month-to-month customers in their first 12 months churn at 51.4% and make up about 55% of all churners. *Action: focus onboarding and retention offers on the first 12 months.*

7. **Senior citizens churn at 41.7%** versus 23.6% for others, and customers without online security churn at 41.8%. Both are useful extra targeting flags.

8. **A simple model can rank who is likely to leave.** The logistic regression reaches a test ROC-AUC of 0.84. Its High-risk tier (>50% predicted probability) holds 1,512 customers who actually churn at 65%, while the Low tier (3,661 customers) churns at only 7%. *Action: send retention offers to the High tier first.*